# 11 · Fill interior gaps against shapely

**Question.** Does Fieldwork's `fill_interior_gaps` widget drop the right rings, and does it
report honestly how much ground the fill adds?

Dropping a polygon's interior rings is exact, so there is no tolerance to argue about. What
the check is really testing is the **bookkeeping**: the count of gaps filled, the area before
net of holes, the area after, and the difference — which is ground the traversal never
computed as reachable and the fill now claims.

The fixture is the 5 and 10 minute corridors from the Broad Street pump. Two bands rather
than one, because the example has a single pump, so one band is one polygon and a one-part
fixture could not tell *filling* from *joining up*.

Criteria fixed before running: the same gap count, no interior ring left, the filled geometry
within 0.05 m² by symmetric difference, and all three areas within 0.05 m².

In [ ]:
# Only what this notebook imports. Requesting rasterio alongside pyproj kills this kernel
# outright; see 00-kernel-check for the measured map.
NEEDED = ["pyproj", "shapely", "numpy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

## Parity

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_11_fill_gaps import run

report = run(Path.cwd() / "fixtures" / "fill-gaps-soho-001.json")
print("criterion:", report["criterion"])
print()
print("agrees:", report["agrees"], "over", report["parts"], "parts")
print(f"gaps filled     fieldwork {report['fieldwork']['gapsFilled']}   shapely {report['reference']['gapsFilled']}")
for key, label in [("areaBeforeSquareMetres", "area before"), ("areaAfterSquareMetres", "area after"),
                   ("areaAddedSquareMetres", "area added ")]:
    print(f"{label}     fieldwork {report['fieldwork'][key]:12.2f}   shapely {report['reference'][key]:12.2f}")
print()
print(f"symmetric difference of the filled geometry {report['geometryDifferenceSquareMetres']:.3e} m2")
print(f"interior rings left in the output           {report['interiorRingsRemaining']}")

## Reading the result

`agrees` means Fieldwork and shapely drop the same rings and account for the same areas. It
says nothing about whether filling is the right thing to do — that is not a question with a
numerical answer, and the reference asserts the filled ground identically, because it is the
same operation.

The widget's own guidance is the useful part: **use the unfilled polygons for coverage, and
the filled ones for a map that should read solid.**